# ex07 · softmax回归的从零实现（对应教材 3.6）

> **做题流程**：按部分补全 TODO，每个自测 cell 跑出 ✓ 再继续；做完再看 `solutions/ex07-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节是**全书第一个分类训练**：复用 ex02 的训练循环骨架 + ex05 的 softmax/交叉熵 + ex06 的数据加载，在 Fashion-MNIST 上把图片分成 10 类，最终测试准确率约 0.85。

In [1]:
import torch
import torchvision
from torch.utils import data
from torchvision import transforms

def load_data_fashion_mnist(batch_size):
    trans = transforms.ToTensor()
    mnist_train = torchvision.datasets.FashionMNIST(
        root='../data', train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root='../data', train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True),
            data.DataLoader(mnist_test, batch_size, shuffle=False))

batch_size = 256
train_iter, test_iter = load_data_fashion_mnist(batch_size)
print('训练集批数:', len(train_iter), ' 测试集批数:', len(test_iter))

训练集批数: 235  测试集批数: 40


## 第一部分 · 初始化参数（TODO 7.1）

每张图展平成 784 维向量，共 10 个类别。先预测再补全：

- W 的形状为什么是 (784, 10)？b 的形状为什么是 (10,)？

每个样本展平成 784 维向量，经 X·W 映射到 10 维（10 个类别的分数），再加 10 维偏置。784×10 的矩阵把「像素」变换成「10 个类别的 logits」。
- 为什么 W 用小的正态随机初始化、b 用 0？（提示：对称性）

W 若全 0（或全相同），10 个输出完全相同，梯度也相同，学不出差异（对称性破坏）；用小的随机值打破对称。b 没有对称性问题，置 0 无妨。
**【你的预测】**

In [2]:
num_inputs = 784
num_outputs = 10

# TODO 7.1: 初始化 W（torch.normal(0, 0.01, size=(num_inputs, num_outputs))）和 b（zeros），
#           两者都要 requires_grad=True
W = torch.normal(0, 0.01, size = (num_inputs, num_outputs), requires_grad = True)
b = torch.zeros((10,), requires_grad = True)


## 第二部分 · softmax、模型、交叉熵（TODO 7.2 ~ 7.4）

这三样在 ex05 都手算/验证过，这里闭卷重写一遍：

- softmax：exp → 每行求和 → 归一化
- net：把 (N, 1, 28, 28) 展平成 (N, 784) 再做线性变换，最后套 softmax
- cross_entropy：用「真实标签作索引」取出正确类的概率，取负对数

先预测：net 里为什么是 reshape(-1, 784)？cross_entropy 里 y_hat[range(len(y_hat)), y] 在做什么？

**【你的预测】**

In [7]:
def softmax(X):
    # TODO 7.2: X_exp = torch.exp(X)；每行求和（keepdim=True）；返回归一化结果
    X_exp = torch.exp(X)
    part = X_exp.sum(axis = 1, keepdim = True)
    return X_exp / part

def net(X):
    # TODO 7.3: softmax(X.reshape(-1, num_inputs) @ W + b)
    return softmax(X.reshape(-1, num_inputs) @ W + b)

def cross_entropy(y_hat, y):
    # TODO 7.4: -torch.log(y_hat[range(len(y_hat)), y])
    return -torch.log(y_hat[range(len(y_hat)), y])

### 自测：完成 TODO 7.2~7.4 后运行

In [8]:
try:
    Xt = torch.tensor([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]])
    p = softmax(Xt)
    assert torch.allclose(p.sum(1), torch.ones(2), atol=1e-6), 'softmax 每行和应=1'
    print('✓ softmax 每行和为 1:', [round(x, 4) for x in p.flatten().tolist()])

    yh = torch.tensor([[0.1, 0.2, 0.7], [0.8, 0.1, 0.1]])
    yy = torch.tensor([2, 0])
    ce = cross_entropy(yh, yy)
    expected = torch.tensor([-torch.log(torch.tensor(0.7)).item(), -torch.log(torch.tensor(0.8)).item()])
    assert torch.allclose(ce, expected, atol=1e-6), f'cross_entropy 不对: {ce.tolist()}'
    print('✓ cross_entropy:', [round(x, 4) for x in ce.tolist()])

    out_shape = list(net(torch.zeros(4, 1, 28, 28)).shape)
    assert out_shape == [4, 10], f'net 输出形状不对: {out_shape}'
    print('✓ net 输出形状:', out_shape)
except NotImplementedError as e:
    print(f'⚠ {e}')
except AssertionError as e:
    print(f'✗ {e}')

✓ softmax 每行和为 1: [0.09, 0.2447, 0.6652, 0.3333, 0.3333, 0.3333]
✓ cross_entropy: [0.3567, 0.2231]
✓ net 输出形状: [4, 10]


## 第三部分 · 准确率与评估（给定代码，读代码）

accuracy 负责「预测对几个」，evaluate_accuracy 负责「在整个数据集上算准确率」。读代码后回答：

- accuracy 里 `y_hat.argmax(axis=1)` 在做什么？为什么能判断预测对不对？
- evaluate_accuracy 为什么包在 `torch.no_grad()` 里？

In [9]:
def accuracy(y_hat, y):
    """预测正确的样本数"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)   # 取每行概率最大的类别
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())


def evaluate_accuracy(net, data_iter):
    """在整个 data_iter 上计算准确率"""
    metric = 0.0
    n = 0
    with torch.no_grad():
        for X, y in data_iter:
            y_hat = net(X)
            metric += accuracy(y_hat, y)
            n += len(X)
    return metric / n

## 第四部分 · 训练循环（TODO 7.5 ~ 7.7）

和 ex02 的循环几乎一样，只是损失换成了交叉熵。补全三处 TODO（考点：交叉熵返回的是向量，反向前要求和成标量）。

先预测：10 个 epoch 后，测试准确率大约到多少？（提示：随机猜是 0.1，训练好大约 0.8x）

**【你的预测】**

In [12]:
def sgd(params, lr, batch_size):
    """小批量随机梯度下降（与 ex02 相同）"""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

lr = 0.1
num_epochs = 10

try:
    for epoch in range(num_epochs):
        train_l_sum = 0.0
        train_acc_sum = 0.0
        n = 0
        for X, y in train_iter:
            y_hat = net(X)
            # TODO 7.5: 计算这个 batch 的交叉熵损失 l
            l = cross_entropy(y_hat, y)
            # TODO 7.6: 反向传播（交叉熵是 (batch,) 向量，先求和成标量）
            l.sum().backward()
            # TODO 7.7: 用 sgd 更新 [W, b]
            sgd([W, b], lr, batch_size)
            train_l_sum += float(l.sum())
            train_acc_sum += accuracy(y_hat, y)
            n += len(X)
        test_acc = evaluate_accuracy(net, test_iter)
        print(f'epoch {epoch + 1}, loss {train_l_sum / n:.4f}, train acc {train_acc_sum / n:.3f}, test acc {test_acc:.3f}')
except NotImplementedError as e:
    print(f'⚠ {e}，先完成 TODO 7.5~7.7 再运行')

epoch 1, loss 0.7815, train acc 0.751, test acc 0.794
epoch 2, loss 0.5703, train acc 0.814, test acc 0.810
epoch 3, loss 0.5244, train acc 0.825, test acc 0.819
epoch 4, loss 0.5009, train acc 0.833, test acc 0.821
epoch 5, loss 0.4849, train acc 0.837, test acc 0.826
epoch 6, loss 0.4740, train acc 0.840, test acc 0.829
epoch 7, loss 0.4649, train acc 0.843, test acc 0.832
epoch 8, loss 0.4580, train acc 0.844, test acc 0.832
epoch 9, loss 0.4527, train acc 0.847, test acc 0.829
epoch 10, loss 0.4469, train acc 0.848, test acc 0.834


## 第五部分 · 结果检验

先预测再运行：训练后的测试准确率、以及「预测错误但概率很高的样本」长什么样。

**【你的预测】**

In [13]:
test_acc = evaluate_accuracy(net, test_iter)
print(f'最终测试准确率: {test_acc:.3f}')

最终测试准确率: 0.834


## 小结与面试衔接

- 分类训练四件套不变：数据 → 模型(softmax) → 损失(交叉熵) → 优化器(sgd)，只是把线性回归的「标量输出」换成「10 维概率输出」
- 交叉熵返回向量，反向前要求和成标量；这与 ex02 的 l.sum() 是同一个道理
- 训练准确率 ≈ 测试准确率，说明没有明显过拟合（ch04 会正式讲诊断）
- 面试高频：softmax 回归 = 线性回归 + softmax + 交叉熵；输出是概率分布，取 argmax 得预测类别